# Phát hiện tàu hàng loạt từ một thư mục Google Drive công khai (ảnh JP2 TCI)

Notebook **độc lập** dùng **model YOLO11 bạn đã fine-tune** (`best.pt`). Nhận **link thư mục Google Drive công khai** chứa nhiều ảnh `.jp2` TCI, **tự tải về**, rồi với **từng ảnh**: cắt tile 320×320 → suy luận → gộp NMS → **xuất kết quả chất lượng cao** (GeoTIFF/JP2 vẽ sẵn box + vector GeoJSON/GPKG), kèm bảng tổng hợp.

**Cấu hình Confidence Threshold** (`CFG["CONF"]`): hạ ~`0.10–0.15` khi **tàu mờ / khó nhìn**; tăng ~`0.4–0.5` khi muốn chắc chắn.

> Thư mục Drive phải để **"Anyone with the link"**. Có thể **(tuỳ chọn) vẽ cả ground truth** (xanh lá) từ GeoPackage Zenodo `10.5281/zenodo.10046341` — lọc theo tên file mỗi ảnh.

## 1. Cấu hình (SỬA TRƯỚC KHI CHẠY)

In [ ]:
import os

CFG = dict(
    # ===== MODEL đã fine-tune của bạn =====
    MODEL_PATH        = "/kaggle/input/my-finetuned-yolo/best.pt",   # <<< SỬA: trỏ tới best.pt của bạn
    HF_FALLBACK_REPO  = "mayrajeo/marine-vessel-yolo",
    HF_FALLBACK_FILE  = "yolo11s_tci.pt",

    # ===== Nguồn ảnh: thư mục Google Drive công khai =====
    DRIVE_FOLDER_URL  = "https://drive.google.com/drive/folders/XXXXXXXXXXXXXXXXXXXXXXXXX",  # <<< SỬA
    LIMIT             = None,   # None = xử lý hết; đặt số (vd 3) để chạy thử vài ảnh đầu

    # ===== Tiling & suy luận =====
    TILE          = 320,
    OVERLAP       = 64,
    IMGSZ         = 320,
    CONF          = 0.25,    # <<< NGƯỠNG CONFIDENCE: giảm 0.10–0.15 khi tàu mờ / khó nhìn
    IOU_NMS       = 0.7,
    NMS_IOU       = 0.5,
    BATCH         = 16,
    SKIP_DARK     = 5,
    DEVICE        = "auto",

    # ===== Xuất kết quả chất lượng cao (giữ georeference) =====
    SAVE_GEOTIFF  = True,    # .tif full-res vẽ box — KHUYẾN NGHỊ
    SAVE_JP2      = False,   # .jp2 full-res vẽ box (đúng định dạng gốc; chậm hơn)
    SAVE_VECTOR   = True,    # vector GeoJSON (lon/lat) + GPKG (CRS gốc)
    BURN_MAX_MP   = 400,     # chặn burn full-res nếu ảnh > ngần này megapixel
    OVERVIEW_MAX  = 1600,

    # ===== Ground truth (tuỳ chọn) — GeoPackage nhãn Zenodo 10.5281/zenodo.10046341 =====
    # Cấu trúc: MỖI TILE 1 FILE '<tile>.gpkg' (vd 34VEM.gpkg), MỖI NGÀY 1 LAYER (vd '20220813').
    # Batch tự khớp từng ảnh theo TILE + NGÀY đọc từ tên file.
    DRAW_GT       = False,   # True để vẽ cả ground truth (xanh lá)
    GT_GPKG_PATH  = "/kaggle/input/zenodo-vessel-gpkg",  # <<< THƯ MỤC chứa các <tile>.gpkg
    GT_LAYER      = None,    # None = tự chọn layer theo NGÀY tên ảnh (nên để None cho batch nhiều ngày)

    DL_DIR        = "/kaggle/working/drive_tci",     # nơi lưu ảnh tải về từ Drive
    OUT_DIR       = "/kaggle/working/drive_infer",   # nơi lưu kết quả
)
os.makedirs(CFG["DL_DIR"], exist_ok=True)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
print("Ngưỡng confidence:", CFG["CONF"])

## 2. Cài đặt thư viện & chọn thiết bị

In [ ]:
# Cài thư viện: ultralytics (model), rasterio (đọc/ghi JP2+GeoTIFF), geopandas (vector + ground truth), gdown
%pip install -q -U ultralytics "huggingface_hub>=0.24" rasterio geopandas gdown

import ultralytics
ultralytics.checks()
print("ultralytics:", ultralytics.__version__)
import rasterio, geopandas
print("rasterio:", rasterio.__version__, "| geopandas:", geopandas.__version__)
# Kiểm tra driver ghi JP2 (JP2OpenJPEG) — cần cho việc xuất .jp2 vẽ sẵn box
_jp2 = rasterio.drivers.raster_driver_extensions().get("jp2", "")
print("JP2 driver:", _jp2 or "(không thấy — vẫn xuất được GeoTIFF/GeoJSON)")

In [ ]:
import torch

def pick_device(pref):
    """auto: dùng GPU nếu chạy được kernel, lỗi (hay gặp P100 trên Kaggle) -> rơi về CPU."""
    if pref not in ("auto", None, ""):
        print("DEVICE ép:", pref); return str(pref)
    if not torch.cuda.is_available():
        print("CUDA không khả dụng -> CPU"); return "cpu"
    try:
        _ = (torch.zeros(16, device="cuda") + 1).sum().item(); torch.cuda.synchronize()
        print("GPU OK:", torch.cuda.get_device_name(0)); return "0"
    except Exception as e:
        print("⚠️ GPU không chạy kernel:", str(e).splitlines()[0], "-> CPU (đổi Accelerator sang T4)")
        return "cpu"

DEVICE_EFF = pick_device(CFG["DEVICE"])
print("DEVICE_EFF =", DEVICE_EFF)

## 3. Nạp model đã fine-tune (best.pt)

In [ ]:
from ultralytics import YOLO

def resolve_model(cfg):
    """Ưu tiên model đã fine-tune của bạn (MODEL_PATH). Nếu không thấy, tải tạm yolo11s_tci.pt từ HF
    để notebook vẫn chạy được (nhưng CẢNH BÁO: đó là model gốc, sai miền dữ liệu của bạn)."""
    p = cfg["MODEL_PATH"]
    if p and os.path.exists(p):
        print("Dùng model đã fine-tune:", p); return p
    print(f"⚠️ Không thấy MODEL_PATH='{p}'. Tải tạm '{cfg['HF_FALLBACK_FILE']}' từ HuggingFace làm dự phòng.")
    print("   -> Nên add best.pt (model bạn đã fine-tune) làm Kaggle Dataset và trỏ MODEL_PATH vào đó.")
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=cfg["HF_FALLBACK_REPO"], filename=cfg["HF_FALLBACK_FILE"],
                           local_dir="/kaggle/working/weights")

MODEL_WEIGHTS = resolve_model(CFG)
model = YOLO(MODEL_WEIGHTS)
print("Đã nạp model:", MODEL_WEIGHTS)

## 4. Hàm xử lý — cắt tile, suy luận, vẽ box, xuất kết quả chất lượng cao

In [ ]:
# =========================================================================
#  Bộ hàm dùng chung: cắt tile -> suy luận -> gộp NMS -> xuất kết quả chất lượng cao
#  (GeoTIFF/JP2 vẽ sẵn box + vector GeoJSON/GPKG) và nạp ground truth từ GeoPackage.
# =========================================================================
import os, json as _json
import numpy as np
import rasterio
from rasterio.windows import Window
from PIL import Image
import torch
from torchvision.ops import nms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D


# ----------------------------------------------------------------- đọc raster
def _band_idx(nbands):
    """Chọn 3 band để dựng ảnh RGB (TCI = 3 band; ảnh 1 band -> nhân 3)."""
    return [1, 2, 3] if nbands >= 3 else [1, 1, 1]


def _to_uint8(arr):
    """Ép mảng về uint8 0..255 (TCI vốn đã 8-bit; phòng khi raster khác kiểu)."""
    if arr.dtype == np.uint8:
        return arr
    a = arr.astype(np.float32)
    return np.clip(a / (float(a.max()) or 1.0) * 255.0, 0, 255).astype(np.uint8)


def _read_rgb_window(src, window):
    """Đọc 1 cửa sổ raster -> mảng uint8 (H, W, 3) RGB (tự đệm mép ngoài biên)."""
    arr = src.read(indexes=_band_idx(src.count), window=window, boundless=True, fill_value=0)
    return _to_uint8(np.transpose(arr, (1, 2, 0)))


def _tile_offsets(size, tile, stride):
    """Sinh offset phủ kín 1 chiều, đảm bảo có tile ôm sát mép cuối."""
    if size <= tile:
        return [0]
    offs = list(range(0, size - tile + 1, stride))
    if offs[-1] != size - tile:
        offs.append(size - tile)
    return offs


# ----------------------------------------------------------------- suy luận tiling
def infer_large_raster(raster_path, model, conf=0.25, tile=320, overlap=64,
                       imgsz=320, iou_nms=0.7, nms_iou=0.5, batch=16,
                       skip_dark=5, device="cpu", verbose=True):
    """Cắt tile -> suy luận từng tile -> dời box về ảnh gốc -> NMS toàn ảnh.

    Trả về: (boxes[N,4] xyxy theo pixel ảnh gốc, scores[N], (W, H))."""
    with rasterio.open(raster_path) as src:
        W, H, nb = src.width, src.height, src.count
        stride = max(1, tile - overlap)
        xs, ys = _tile_offsets(W, tile, stride), _tile_offsets(H, tile, stride)
        if verbose:
            print(f"Ảnh {W}×{H}px, {nb} band -> {len(xs)}×{len(ys)} = {len(xs) * len(ys)} tile "
                  f"(tile={tile}, overlap={overlap}, imgsz={imgsz}).")

        boxes_all, scores_all = [], []
        buf_imgs, buf_off = [], []

        def _flush():
            if not buf_imgs:
                return
            for res, (ox, oy) in zip(
                    model.predict(buf_imgs, imgsz=imgsz, conf=conf, iou=iou_nms,
                                  device=device, verbose=False), buf_off):
                if res.boxes is None or len(res.boxes) == 0:
                    continue
                xyxy = res.boxes.xyxy.cpu().numpy().copy()
                xyxy[:, [0, 2]] += ox        # dời toạ độ tile -> toạ độ ảnh gốc
                xyxy[:, [1, 3]] += oy
                boxes_all.append(xyxy)
                scores_all.append(res.boxes.conf.cpu().numpy())
            buf_imgs.clear(); buf_off.clear()

        n_used = 0
        for oy in ys:
            for ox in xs:
                arr = _read_rgb_window(src, Window(ox, oy, tile, tile))
                if int(arr.max()) < skip_dark:          # tile đen (nodata) -> bỏ
                    continue
                buf_imgs.append(Image.fromarray(arr)); buf_off.append((ox, oy)); n_used += 1
                if len(buf_imgs) >= batch:
                    _flush()
        _flush()

    if verbose:
        print(f"Đã suy luận {n_used} tile có dữ liệu.")
    if not boxes_all:
        return np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), (W, H)

    boxes = np.concatenate(boxes_all).astype(np.float32)
    scores = np.concatenate(scores_all).astype(np.float32)
    keep = nms(torch.from_numpy(boxes), torch.from_numpy(scores), nms_iou).numpy()  # NMS toàn ảnh
    return boxes[keep], scores[keep], (W, H)


# ----------------------------------------------------------------- xem nhanh (matplotlib)
def draw_overview(raster_path, boxes, gt_boxes=None, max_side=1600, title=None,
                  save_path=None, show=True):
    """Vẽ box lên ảnh thu nhỏ (overview) để nhìn phân bố. Đỏ = dự đoán, xanh lá = ground truth."""
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        scale = min(1.0, max_side / max(W, H))
        ow, oh = max(1, int(W * scale)), max(1, int(H * scale))
        ov = _to_uint8(np.transpose(
            src.read(indexes=_band_idx(src.count), out_shape=(3, oh, ow)), (1, 2, 0)))
    fig, ax = plt.subplots(figsize=(13, max(3, 13 * oh / ow)))
    ax.imshow(ov)
    if gt_boxes is not None and len(gt_boxes):
        for (x1, y1, x2, y2) in (np.asarray(gt_boxes) * scale):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="lime", linewidth=1.0))
    for (x1, y1, x2, y2) in (np.asarray(boxes) * scale):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor="red", linewidth=0.8))
    handles = [Line2D([0], [0], color="red", lw=2, label="Dự đoán")]
    if gt_boxes is not None and len(gt_boxes):
        handles.append(Line2D([0], [0], color="lime", lw=2, label="Ground truth"))
    ax.legend(handles=handles, loc="upper right", fontsize=9)
    ax.set_title(title or f"{os.path.basename(raster_path)} — {len(boxes)} dự đoán", fontsize=12)
    ax.axis("off")
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show() if show else plt.close(fig)


def show_detection_crops(raster_path, boxes, scores, topk=12, pad=40, ncol=4,
                         save_path=None, show=True):
    """Cắt cận cảnh top-K tàu (theo confidence) ở ĐỘ PHÂN GIẢI GỐC để mắt thường kiểm chứng."""
    if len(boxes) == 0:
        print("Không có tàu nào để cắt cận cảnh."); return
    order = np.argsort(-scores)[:topk]
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        nrow = (len(order) + ncol - 1) // ncol
        fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3 * nrow))
        axes = np.atleast_1d(axes).flatten()
        for ax, i in zip(axes, order):
            x1, y1, x2, y2 = boxes[i]
            cx0, cy0 = max(0, int(x1) - pad), max(0, int(y1) - pad)
            cw = min(W - cx0, int(x2 - x1) + 2 * pad)
            ch = min(H - cy0, int(y2 - y1) + 2 * pad)
            ax.imshow(_read_rgb_window(src, Window(cx0, cy0, cw, ch)))
            ax.add_patch(patches.Rectangle((x1 - cx0, y1 - cy0), x2 - x1, y2 - y1,
                                           fill=False, edgecolor="red", linewidth=1.5))
            ax.set_title(f"conf={scores[i]:.2f}", fontsize=9); ax.axis("off")
        for ax in axes[len(order):]:
            ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show() if show else plt.close(fig)


# ----------------------------------------------------------------- xuất KẾT QUẢ CHẤT LƯỢNG CAO
def burn_boxes_to_raster(raster_path, out_path, pred_boxes, pred_scores=None, gt_boxes=None,
                         driver="GTiff", thickness=None, jp2_quality=100, max_megapixels=400):
    """VẼ box vào ẢNH Ở ĐỘ PHÂN GIẢI GỐC rồi ghi ra file GIỮ NGUYÊN georeference (CRS + transform).

    - driver="GTiff"       -> .tif (khuyến nghị: nhanh, nén LZW không mất dữ liệu, mở tốt ở QGIS).
    - driver="JP2OpenJPEG" -> .jp2 (đúng định dạng gốc; jp2_quality=100 -> không mất dữ liệu, chậm hơn).
    Đỏ = dự đoán, xanh lá = ground truth. Cần đủ RAM (đọc cả ảnh) — chặn nếu quá max_megapixels."""
    import cv2
    with rasterio.open(raster_path) as src:
        W, H = src.width, src.height
        mp = W * H / 1e6
        if mp > max_megapixels:
            raise MemoryError(
                f"Ảnh {W}×{H} ({mp:.0f} MP) vượt max_megapixels={max_megapixels}. "
                "Vẽ full-res sẽ tốn RAM — tăng max_megapixels nếu chắc đủ RAM, hoặc dùng vector (GeoJSON/GPKG).")
        img = np.ascontiguousarray(_read_rgb_window(src, Window(0, 0, W, H)))  # (H, W, 3) RGB uint8
        crs, transform = src.crs, src.transform

    t = int(thickness or max(2, round(max(W, H) / 2000)))

    def _draw(boxes, color):        # color theo RGB (ảnh đang là RGB)
        if boxes is None:
            return
        for b in np.asarray(boxes):
            x1, y1, x2, y2 = (int(round(v)) for v in b[:4])
            cv2.rectangle(img, (x1, y1), (x2, y2), color, t)

    _draw(gt_boxes, (0, 255, 0))     # ground truth: xanh lá (vẽ trước)
    _draw(pred_boxes, (255, 0, 0))   # dự đoán: đỏ (đè lên)

    profile = dict(driver=driver, width=W, height=H, count=3, dtype="uint8",
                   crs=crs, transform=transform)
    if driver == "GTiff":
        profile.update(compress="LZW", tiled=True, blockxsize=512, blockysize=512,
                       photometric="RGB", BIGTIFF="IF_SAFER")
    elif driver == "JP2OpenJPEG":
        profile.update(QUALITY=jp2_quality, REVERSIBLE=("YES" if jp2_quality >= 100 else "NO"))
    with rasterio.open(out_path, "w", **profile) as dst:
        for k in range(3):
            dst.write(img[:, :, k], k + 1)   # ghi từng band, tránh copy transpose tốn RAM
    print(f"  ✔ Đã ghi ảnh vẽ sẵn box (giữ georef): {out_path}")
    return out_path


def detections_to_vectors(raster_path, boxes, scores, out_geojson=None, out_gpkg=None):
    """Xuất box dự đoán thành VECTOR georef (overlay hoàn hảo lên .jp2 gốc trong QGIS, không mất pixel).

    - .gpkg giữ nguyên CRS gốc của ảnh; .geojson đưa về lon/lat (EPSG:4326) cho web/chia sẻ."""
    try:
        import geopandas as gpd
        from shapely.geometry import box as shp_box
    except Exception as e:
        print("  (Bỏ qua xuất vector — thiếu geopandas/shapely:", str(e).splitlines()[0], ")")
        return None
    with rasterio.open(raster_path) as src:
        T, crs = src.transform, src.crs
    geoms, confs = [], []
    for (x1, y1, x2, y2), s in zip(boxes.tolist(), scores.tolist()):
        (X1, Y1) = rasterio.transform.xy(T, y1, x1)   # (row=y, col=x) -> (x_crs, y_crs)
        (X2, Y2) = rasterio.transform.xy(T, y2, x2)
        geoms.append(shp_box(min(X1, X2), min(Y1, Y2), max(X1, X2), max(Y1, Y2)))
        confs.append(round(float(s), 4))
    gdf = gpd.GeoDataFrame({"confidence": confs, "class": "vessel"}, geometry=geoms, crs=crs)
    if out_gpkg:
        gdf.to_file(out_gpkg, driver="GPKG"); print("  ✔ Vector GPKG (CRS gốc):", out_gpkg)
    if out_geojson:
        (gdf.to_crs(4326) if crs is not None else gdf).to_file(out_geojson, driver="GeoJSON")
        print("  ✔ Vector GeoJSON (lon/lat):", out_geojson)
    return gdf


def save_detections_json(raster_path, boxes, scores, out_json):
    """Lưu danh sách box (pixel) ra JSON; kèm toạ độ tâm theo CRS gốc nếu ảnh có georef."""
    crs, recs = None, []
    try:
        with rasterio.open(raster_path) as src:
            T, crs = src.transform, src.crs
            for (x1, y1, x2, y2), s in zip(boxes.tolist(), scores.tolist()):
                rec = {"confidence": round(float(s), 4),
                       "bbox_xyxy_pixel": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]}
                if crs is not None:
                    gx, gy = rasterio.transform.xy(T, (y1 + y2) / 2.0, (x1 + x2) / 2.0)
                    rec["center_crs_xy"] = [round(gx, 2), round(gy, 2)]
                recs.append(rec)
    except Exception as e:
        print("Bỏ qua georef:", str(e).splitlines()[0])
        recs = [{"confidence": round(float(s), 4),
                 "bbox_xyxy_pixel": [round(float(v), 1) for v in b]}
                for b, s in zip(boxes.tolist(), scores.tolist())]
    out = {"raster": os.path.basename(raster_path), "num_detections": len(recs),
           "crs": (str(crs) if crs is not None else None), "detections": recs}
    with open(out_json, "w") as f:
        _json.dump(out, f, indent=2)
    return out


# ----------------------------------------------------------------- GROUND TRUTH từ GeoPackage (Zenodo)
def _list_layers(gpkg_path):
    """Liệt kê layer trong GeoPackage, không phụ thuộc engine (pyogrio hoặc fiona)."""
    try:
        import pyogrio
        return [l[0] for l in pyogrio.list_layers(gpkg_path)]
    except Exception:
        pass
    try:
        import fiona
        return list(fiona.listlayers(gpkg_path))
    except Exception:
        return [None]     # không liệt kê được -> đọc layer mặc định


def inspect_gpkg(gpkg_path):
    """In các layer + cột + CRS + vài dòng mẫu của GeoPackage để biết cột nào chỉ định sản phẩm S2."""
    import geopandas as gpd
    layers = _list_layers(gpkg_path)
    print("GeoPackage:", gpkg_path, "| layers:", layers)
    for lyr in layers:
        g = gpd.read_file(gpkg_path, layer=lyr) if lyr else gpd.read_file(gpkg_path)
        print(f"\n== layer '{lyr}' == n={len(g)} | CRS={g.crs}")
        print("cột:", list(g.columns))
        with_no_geom = [c for c in g.columns if c != g.geometry.name]
        if with_no_geom:
            print(g[with_no_geom].head(3).to_string())
    return layers


def load_gt_boxes_from_gpkg(gpkg_path, raster_path, product_col=None, product_value=None,
                            layer=None, verbose=True):
    """Nạp ground-truth box từ GeoPackage của Zenodo -> trả về boxes[N,4] theo pixel ảnh.

    - product_col / product_value: lọc annotation đúng cảnh S2 (khớp 'chứa', không phân biệt hoa/thường).
      Nếu để None sẽ dùng TẤT CẢ annotation trong file (chỉ nên vậy nếu GPKG chỉ chứa 1 cảnh).
    - Tự chuyển CRS của annotation về CRS của ảnh rồi đổi sang toạ độ pixel."""
    import geopandas as gpd
    gdf = gpd.read_file(gpkg_path, layer=layer) if layer else gpd.read_file(gpkg_path)
    if verbose:
        print(f"GT gốc: {len(gdf)} annotation | CRS={gdf.crs} | cột={list(gdf.columns)}")
    if product_col and product_value is not None:
        assert product_col in gdf.columns, \
            f"Không có cột '{product_col}'. Các cột hiện có: {list(gdf.columns)}"
        m = gdf[product_col].astype(str).str.contains(str(product_value), case=False, na=False)
        gdf = gdf[m]
        if verbose:
            print(f"Sau lọc {product_col} ~ '{product_value}': {len(gdf)} annotation")
    with rasterio.open(raster_path) as src:
        crs, W, H = src.crs, src.width, src.height
        if crs is not None and gdf.crs is not None and str(gdf.crs) != str(crs):
            gdf = gdf.to_crs(crs)
        boxes = []
        for geom in gdf.geometry:
            if geom is None or geom.is_empty:
                continue
            minx, miny, maxx, maxy = geom.bounds
            r1, c1 = src.index(minx, maxy)     # góc trên-trái (maxy = phía bắc/đỉnh)
            r2, c2 = src.index(maxx, miny)     # góc dưới-phải
            x1, x2 = sorted((c1, c2)); y1, y2 = sorted((r1, r2))
            if x2 < 0 or y2 < 0 or x1 > W or y1 > H:   # nằm ngoài ảnh -> bỏ
                continue
            boxes.append([max(0, x1), max(0, y1), min(W, x2), min(H, y2)])
    if verbose:
        print(f"-> {len(boxes)} GT box nằm trong ảnh.")
    return np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), np.float32)


import re as _re
import glob as _glob


def parse_tile_date(name):
    """Tách (tile_id, date) từ tên file ảnh Sentinel-2.
    Ví dụ 'T34VEM_20220813T0959_TCI.jp2' -> ('34VEM', '20220813')."""
    base = os.path.basename(str(name))
    mt = _re.search(r"T?(\d{2}[A-Za-z]{3})", base)      # ô MGRS: 2 số + 3 chữ (vd 34VEM)
    md = _re.search(r"(?<!\d)(\d{8})(?!\d)", base)       # ngày YYYYMMDD
    return (mt.group(1).upper() if mt else None), (md.group(1) if md else None)


def _resolve_gpkg(gpkg_source, tile):
    """gpkg_source là 1 file .gpkg -> dùng luôn; là THƯ MỤC -> tìm '<tile>.gpkg' theo tile của ảnh."""
    if not os.path.isdir(gpkg_source):
        return gpkg_source
    cands = sorted(_glob.glob(os.path.join(gpkg_source, "*.gpkg")))
    if not tile:
        raise FileNotFoundError(
            f"Không đọc được tile từ tên ảnh. Đặt GT_GPKG_PATH thẳng tới 1 file .gpkg. "
            f"Các file có sẵn: {[os.path.basename(c) for c in cands]}")
    for c in cands:
        stem = os.path.splitext(os.path.basename(c))[0].upper().lstrip("T")
        if stem == tile.upper().lstrip("T"):
            return c
    raise FileNotFoundError(f"Không thấy GPKG cho tile '{tile}' trong {gpkg_source}. "
                            f"Có: {[os.path.basename(c) for c in cands]}")


def _resolve_layer(gpkg, date, layer):
    """Chọn layer: ưu tiên 'layer' do người dùng đặt; nếu None thì khớp layer trùng NGÀY của ảnh;
    nếu file chỉ có 1 layer thì lấy layer đó; nếu nhiều layer mà không rõ ngày -> báo lỗi rõ ràng."""
    if layer:
        return str(layer)
    lyrs = [str(l) for l in _list_layers(gpkg) if l]
    if date and date in lyrs:
        return date
    if len(lyrs) == 1:
        return lyrs[0]
    raise ValueError(
        f"'{os.path.basename(gpkg)}' có nhiều layer (mỗi layer = 1 ngày): {lyrs}. "
        f"Đặt GT_LAYER đúng ngày của ảnh (vd '{lyrs[-1]}').")


def gt_boxes_for_raster(raster_path, gpkg_source, layer=None, verbose=True):
    """Tự khớp ground truth cho 1 ảnh theo cấu trúc Zenodo (1 GPKG / 1 tile, 1 layer / 1 ngày).

    - gpkg_source: thư mục chứa các '<tile>.gpkg', HOẶC 1 file .gpkg cụ thể.
    - layer: None = tự chọn layer theo ngày trong tên ảnh.
    Trả về boxes[N,4] theo pixel ảnh."""
    tile, date = parse_tile_date(raster_path)
    gpkg = _resolve_gpkg(gpkg_source, tile)
    lyr = _resolve_layer(gpkg, date, layer)
    if verbose:
        print(f"GT: tile={tile} date={date} -> {os.path.basename(gpkg)} [layer '{lyr}']")
    return load_gt_boxes_from_gpkg(gpkg, raster_path, layer=lyr, verbose=verbose)


print("Đã nạp helpers: infer_large_raster / draw_overview / show_detection_crops /")
print("               burn_boxes_to_raster / detections_to_vectors / save_detections_json /")
print("               inspect_gpkg / load_gt_boxes_from_gpkg / gt_boxes_for_raster")

## 5. Tải ảnh từ thư mục Google Drive công khai

In [ ]:
import gdown, glob

# Tải toàn bộ thư mục Drive công khai về DL_DIR
gdown.download_folder(CFG["DRIVE_FOLDER_URL"], output=CFG["DL_DIR"],
                      quiet=False, use_cookies=False, remaining_ok=True)

# Gom các file raster tải được (jp2 là chính; thêm vài đuôi phổ biến cho linh hoạt)
RASTER_EXTS = (".jp2", ".tif", ".tiff", ".png", ".jpg", ".jpeg")
raster_files = sorted(p for p in glob.glob(os.path.join(CFG["DL_DIR"], "**", "*"), recursive=True)
                      if p.lower().endswith(RASTER_EXTS))
if CFG["LIMIT"]:
    raster_files = raster_files[:CFG["LIMIT"]]

print(f"\nTải xong. Có {len(raster_files)} ảnh để suy luận:")
for p in raster_files:
    print(f"  - {os.path.relpath(p, CFG['DL_DIR'])}  ({os.path.getsize(p) / 1e6:.1f} MB)")
assert raster_files, "Không tìm thấy ảnh nào — kiểm tra link thư mục có công khai và đúng dạng không."

# (Tuỳ chọn) Nếu bật DRAW_GT: liệt kê các file <tile>.gpkg; xem cấu trúc 1 file bằng inspect_gpkg(<file>).
if CFG["DRAW_GT"]:
    if os.path.isdir(CFG["GT_GPKG_PATH"]):
        gpkgs = sorted(glob.glob(os.path.join(CFG["GT_GPKG_PATH"], "*.gpkg")))
        print(f"\nGeoPackage nhãn ({len(gpkgs)} tile):", [os.path.basename(g) for g in gpkgs])
        if gpkgs:
            inspect_gpkg(gpkgs[0])   # xem layer (ngày) + cột của 1 file mẫu
    else:
        inspect_gpkg(CFG["GT_GPKG_PATH"])

## 5b. Suy luận hàng loạt + xuất kết quả chất lượng cao (mỗi ảnh)

In [ ]:
import pandas as pd
from IPython.display import display

def _gt_for(path):
    """Nạp GT cho 1 ảnh: tự khớp <tile>.gpkg + layer theo NGÀY đọc từ tên file."""
    if not CFG["DRAW_GT"]:
        return None
    try:
        return gt_boxes_for_raster(path, CFG["GT_GPKG_PATH"], layer=CFG["GT_LAYER"], verbose=False)
    except Exception as e:
        print("  (GT lỗi, bỏ qua:", str(e).splitlines()[0], ")"); return None

summary = []
for p in raster_files:
    stem = os.path.splitext(os.path.basename(p))[0]
    print("\n" + "=" * 70 + f"\n>>> {os.path.basename(p)}")
    try:
        b, s, (W, H) = infer_large_raster(
            p, model, conf=CFG["CONF"], tile=CFG["TILE"], overlap=CFG["OVERLAP"],
            imgsz=CFG["IMGSZ"], iou_nms=CFG["IOU_NMS"], nms_iou=CFG["NMS_IOU"],
            batch=CFG["BATCH"], skip_dark=CFG["SKIP_DARK"], device=DEVICE_EFF)
    except Exception as e:
        print("  ⚠️ Lỗi khi suy luận, bỏ qua:", str(e).splitlines()[0])
        summary.append({"file": os.path.basename(p), "num_ships": None, "num_gt": None, "size_MB": None})
        continue

    gt = _gt_for(p)
    print(f"  -> {len(b)} tàu (conf≥{CFG['CONF']})" + (f", {len(gt)} GT" if gt is not None else ""))

    # JSON + vector (chất lượng cao nhất, overlay lên .jp2 gốc)
    save_detections_json(p, b, s, os.path.join(CFG["OUT_DIR"], f"{stem}_detections.json"))
    if CFG["SAVE_VECTOR"]:
        detections_to_vectors(p, b, s,
                              out_geojson=os.path.join(CFG["OUT_DIR"], f"{stem}_pred.geojson"),
                              out_gpkg=os.path.join(CFG["OUT_DIR"], f"{stem}_pred.gpkg"))
    # Ảnh full-res vẽ sẵn box, giữ georef
    if CFG["SAVE_GEOTIFF"]:
        try:
            burn_boxes_to_raster(p, os.path.join(CFG["OUT_DIR"], f"{stem}_annotated.tif"),
                                 b, s, gt_boxes=gt, driver="GTiff", max_megapixels=CFG["BURN_MAX_MP"])
        except Exception as e:
            print("  (GeoTIFF lỗi:", str(e).splitlines()[0], ")")
    if CFG["SAVE_JP2"]:
        try:
            burn_boxes_to_raster(p, os.path.join(CFG["OUT_DIR"], f"{stem}_annotated.jp2"),
                                 b, s, gt_boxes=gt, driver="JP2OpenJPEG",
                                 jp2_quality=100, max_megapixels=CFG["BURN_MAX_MP"])
        except Exception as e:
            print("  (JP2 lỗi:", str(e).splitlines()[0], ")")
    # Xem nhanh
    draw_overview(p, b, gt_boxes=gt, max_side=CFG["OVERVIEW_MAX"],
                  title=f"{stem} — {len(b)} dự đoán (conf≥{CFG['CONF']})",
                  save_path=os.path.join(CFG["OUT_DIR"], f"{stem}_overview.png"))
    show_detection_crops(p, b, s, topk=8,
                         save_path=os.path.join(CFG["OUT_DIR"], f"{stem}_crops.png"))

    summary.append({"file": os.path.basename(p), "num_ships": int(len(b)),
                    "num_gt": (int(len(gt)) if gt is not None else None),
                    "size_MB": round(os.path.getsize(p) / 1e6, 1)})

print("\n===== TỔNG HỢP =====")
df = pd.DataFrame(summary)
display(df)
df.to_csv(os.path.join(CFG["OUT_DIR"], "summary.csv"), index=False)
print("Kết quả (ảnh vẽ box + vector + JSON) đã lưu tại:", CFG["OUT_DIR"])

## 6. Ghi chú

- **Kết quả mỗi ảnh** (trong `/kaggle/working/drive_infer/`):
  - `*_annotated.tif` — ảnh full-res **vẽ sẵn box, giữ georeference** (🔴 dự đoán, 🟢 ground truth). Mở QGIS đúng toạ độ.
  - `*_pred.geojson` (lon/lat) + `*_pred.gpkg` (CRS gốc) — **vector overlay** lên `.jp2` gốc, không đụng pixel (chất lượng cao nhất).
  - `*_detections.json` — toạ độ + confidence; `*_overview.png`, `*_crops.png` — xem nhanh; `summary.csv` — bảng tổng hợp.
- **Tàu khó nhìn:** giảm `CFG["CONF"]` xuống 0.10–0.15 để tăng recall (đổi lại dễ báo nhầm hơn).
- **Ground truth:** đặt các file GeoPackage Zenodo (`10.5281/zenodo.10046341`) — **mỗi tile 1 file** `<tile>.gpkg` (vd `34VEM.gpkg`), **mỗi ngày 1 layer** (vd `20220813`) — làm Kaggle Dataset, trỏ `CFG["GT_GPKG_PATH"]` tới **thư mục** đó và bật `CFG["DRAW_GT"]=True`. Batch **tự khớp** từng ảnh theo **tile + ngày** đọc từ tên file (`T34VEM_20220813_..._TCI.jp2` → `34VEM.gpkg`, layer `20220813`). Vì vậy **tên file ảnh phải chứa mã tile và ngày**. Nếu không có ngày trong tên, đặt `CFG["GT_LAYER"]` thủ công.
- **gdown giới hạn thư mục lớn:** nếu thư mục Drive > 50 file, `remaining_ok=True` đã bật để tải hết; nếu vẫn thiếu, chia nhỏ thư mục.
- **Hết RAM khi burn ảnh lớn:** tăng `BURN_MAX_MP` hoặc tắt `SAVE_GEOTIFF/SAVE_JP2` và chỉ dùng vector.
- **Cách đọc GT khớp geo2ml của tác giả** (`gpkg_layer=<ngày>`, `target_column='id'`, `gridsize=320`, `ann_format='box'`), nên box GT vẽ ra trùng nhãn gốc. Muốn **đo định lượng** (P/R/mAP) thì dùng `geo2ml.create_yolo_dataset(...)` để cắt bộ eval rồi `YOLO(MODEL_WEIGHTS).val(...)`.